# 01 - Data Understanding and Baseline Cleaning
## Surprise Medical Bill Risk Predictor - SDSU Capstone

This notebook:
1. Generates realistic CMS-style healthcare provider data
2. Performs data quality checks and cleaning
3. Outputs `data/base_cleaned.parquet` for feature engineering

**Data Source:** Synthetic CMS (Centers for Medicare & Medicaid Services) style data
- Inpatient Prospective Payment System (IPPS) data structure
- Outpatient Prospective Payment System (OPPS) data structure
- ~100,000 provider-service records across all 50 states

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

os.makedirs('data', exist_ok=True)

print('Libraries loaded successfully')
print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')

## 1. Generate Synthetic CMS Data

Generating realistic provider data based on CMS data structure and known distributions.

In [ ]:
np.random.seed(42)

# === CONFIGURATION ===
N_RECORDS = 100_000
N_PROVIDERS = 600

# All 50 US states
states = [
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA',
    'HI','ID','IL','IN','IA','KS','KY','LA','ME','MD',
    'MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
    'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC',
    'SD','TN','TX','UT','VT','VA','WA','WV','WI','WY'
]

# DRG definitions (common inpatient diagnoses)
drg_definitions = [
    '470 - MAJOR JOINT REPLACEMENT OR REATTACHMENT OF LOWER EXTREMITY',
    '871 - SEPTICEMIA OR SEVERE SEPSIS W/O MV >96 HOURS',
    '291 - HEART FAILURE & SHOCK W MCC',
    '392 - ESOPHAGITIS, GASTROENT & MISC DIGEST DISORDERS W/O MCC',
    '194 - SIMPLE PNEUMONIA & PLEURISY W CC',
    '641 - MISC DISORDERS OF NUTRITION, METABOLISM, FLUIDS/ELECTROLYTES',
    '682 - RENAL FAILURE W MCC',
    '378 - G.I. HEMORRHAGE W CC',
    '065 - INTRACRANIAL HEMORRHAGE OR CEREBRAL INFARCTION W CC',
    '603 - CELLULITIS W/O MCC',
    '252 - OTHER VASCULAR PROCEDURES W CC',
    '481 - HIP & FEMUR PROCEDURES EXCEPT MAJOR JOINT W CC',
    '247 - PERC CARDIOVASC PROC W DRUG-ELUTING STENT W/O MCC',
    '189 - PULMONARY EDEMA & RESPIRATORY FAILURE',
    '536 - FRACTURES OF HIP & PELVIS W/O MCC',
    '101 - SEIZURES W/O MCC',
    '690 - KIDNEY & URINARY TRACT INFECTIONS W/O MCC',
    '313 - CHEST PAIN',
    '775 - VAGINAL DELIVERY W/O COMPLICATING DIAGNOSES',
    '807 - VAGINAL DELIVERY W STERILIZATION &/OR D&C',
    '311 - ANGINA PECTORIS',
    '280 - ACUTE MYOCARDIAL INFARCTION, DISCHARGED ALIVE W MCC',
    '460 - SPINAL FUSION EXCEPT CERVICAL W/O MCC',
    '149 - DYSEQUILIBRIUM',
    '552 - MEDICAL BACK PROBLEMS W/O MCC',
    '897 - ALCOHOL/DRUG ABUSE OR DEPENDENCE W/O REHABILITATION',
    '621 - O.R. PROCEDURES FOR OBESITY W/O CC/MCC',
    '372 - MAJOR GASTROINTESTINAL DISORDERS & PERITONEAL INFECTIONS',
    '069 - TRANSIENT ISCHEMIA',
    '057 - DEGENERATIVE NERVOUS SYSTEM DISORDERS W/O MCC',
]

# Service types mapped to DRG categories
service_type_map = {
    '470': 'inpatient surgery', '252': 'inpatient surgery', '481': 'inpatient surgery',
    '247': 'inpatient surgery', '460': 'inpatient surgery', '621': 'inpatient surgery',
    '871': 'emergency department', '291': 'emergency department', '280': 'emergency department',
    '189': 'emergency department', '065': 'emergency department',
    '194': 'outpatient surgery', '392': 'outpatient surgery', '682': 'outpatient surgery',
    '378': 'outpatient surgery', '603': 'outpatient surgery', '536': 'outpatient surgery',
    '641': 'same-day procedure', '313': 'same-day procedure', '311': 'same-day procedure',
    '101': 'same-day procedure', '690': 'same-day procedure', '149': 'same-day procedure',
    '775': 'same-day procedure', '807': 'same-day procedure',
    '552': 'imaging', '069': 'imaging', '057': 'imaging',
    '897': 'imaging', '372': 'imaging',
}

print(f'Configuration set: {N_RECORDS:,} records, {N_PROVIDERS} providers, {len(states)} states')

In [ ]:
# === GENERATE PROVIDER MASTER LIST ===
print('Generating provider master list...')

hospital_prefixes = ['St.', 'General', 'Memorial', 'Regional', 'Community', 'University',
                     'Methodist', 'Baptist', 'Presbyterian', 'Mercy', 'Sacred Heart',
                     'Valley', 'Riverside', 'Lakeside', 'Mountain View']
hospital_suffixes = ['Hospital', 'Medical Center', 'Health System', 'Healthcare', 'Clinic']

# State-based risk profile (some states have higher surprise billing rates)
state_risk_profiles = {s: np.random.uniform(0.3, 0.8) for s in states}
# Known high-risk states get higher risk
for s in ['TX', 'FL', 'NY', 'NJ', 'GA', 'AZ', 'NC', 'VA']:
    state_risk_profiles[s] = np.random.uniform(0.55, 0.85)

providers = []
for i in range(N_PROVIDERS):
    state = np.random.choice(states, p=None)
    prefix = np.random.choice(hospital_prefixes)
    suffix = np.random.choice(hospital_suffixes)
    city_name = f'City{i % 200 + 1}'
    providers.append({
        'provider_id': 100000 + i,
        'provider_name': f'{prefix} {city_name} {suffix}',
        'provider_state': state,
        # Provider-specific risk multiplier (some hospitals overbill more)
        'provider_risk_multiplier': np.random.lognormal(0, 0.4),
    })

providers_df = pd.DataFrame(providers)
print(f'Generated {len(providers_df)} providers across {providers_df["provider_state"].nunique()} states')

In [ ]:
# === GENERATE RECORDS ===
print('Generating healthcare records...')

records = []
drg_codes = [d.split(' - ')[0].strip() for d in drg_definitions]
drg_weights = np.array([0.08, 0.07, 0.07, 0.06, 0.06, 0.05, 0.05, 0.05, 0.04, 0.04,
                         0.04, 0.04, 0.04, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.02,
                         0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02])
drg_weights /= drg_weights.sum()

# DRG-specific cost parameters (mean inpatient charges)
drg_cost_params = {
    '470': (85000, 25000), '252': (75000, 20000), '481': (55000, 15000),
    '247': (65000, 18000), '460': (95000, 28000), '621': (45000, 12000),
    '871': (55000, 18000), '291': (48000, 14000), '280': (70000, 22000),
    '189': (42000, 12000), '065': (52000, 16000),
    '194': (32000, 9000), '392': (28000, 8000), '682': (45000, 14000),
    '378': (38000, 11000), '603': (25000, 7000), '536': (35000, 10000),
    '641': (22000, 6000), '313': (18000, 5000), '311': (20000, 6000),
    '101': (24000, 7000), '690': (19000, 5500), '149': (15000, 4500),
    '775': (16000, 5000), '807': (18000, 5500),
    '552': (12000, 3500), '069': (14000, 4000), '057': (11000, 3200),
    '897': (9000, 2800), '372': (30000, 9000),
}

prov_indices = np.random.choice(len(providers_df), size=N_RECORDS, replace=True)
drg_indices = np.random.choice(len(drg_codes), size=N_RECORDS, replace=True, p=drg_weights)

for i in range(N_RECORDS):
    prov = providers_df.iloc[prov_indices[i]]
    drg_code = drg_codes[drg_indices[i]]
    drg_def = drg_definitions[drg_indices[i]]
    
    state = prov['provider_state']
    state_risk = state_risk_profiles[state]
    prov_mult = prov['provider_risk_multiplier']
    
    # Base costs from DRG parameters
    base_mean, base_std = drg_cost_params.get(drg_code, (30000, 9000))
    
    # Inpatient data
    avg_covered = max(5000, np.random.normal(base_mean * prov_mult, base_std))
    # Payment is always less than covered charges
    payment_ratio = np.random.beta(3, 2) * 0.6  # Typically 20-60% of covered charges
    avg_total_payments = avg_covered * payment_ratio
    avg_medicare_payment = avg_total_payments * np.random.uniform(0.7, 0.95)
    total_discharges = max(1, int(np.random.lognormal(3.5, 0.8)))
    
    # Outpatient billing data
    billed_amount = avg_covered * np.random.uniform(0.05, 0.20) * prov_mult
    medicare_payment = billed_amount * np.random.beta(2, 3) * 0.5
    
    service_type = service_type_map.get(drg_code, 'outpatient surgery')
    
    records.append({
        'provider_id': prov['provider_id'],
        'provider_name': prov['provider_name'],
        'provider_state': state,
        'DRG Definition': drg_def,
        'drg_code': drg_code,
        'service_type': service_type,
        'Total Discharges': total_discharges,
        'average_covered_charges': round(avg_covered, 2),
        'average_total_payments': round(avg_total_payments, 2),
        'average_medicare_payments': round(avg_medicare_payment, 2),
        'Billed amount': round(billed_amount, 2),
        'Medicare payment': round(medicare_payment, 2),
    })

raw_df = pd.DataFrame(records)
print(f'Generated {len(raw_df):,} records')
print(f'Unique providers: {raw_df["provider_id"].nunique()}')
print(f'Unique states: {raw_df["provider_state"].nunique()}')
print(f'Service types: {raw_df["service_type"].value_counts().to_dict()}')

## 2. Data Quality Checks

In [ ]:
print('=== DATA QUALITY REPORT ===')
print(f'\nShape: {raw_df.shape}')
print(f'\nMissing values:')
print(raw_df.isnull().sum())
print(f'\nData types:')
print(raw_df.dtypes)
print(f'\nNumeric summary:')
print(raw_df[['average_covered_charges','average_total_payments','Billed amount','Medicare payment']].describe())

In [ ]:
# === CLEANING STEPS ===
print('Cleaning data...')
df_clean = raw_df.copy()

# 1. Remove records with zero or negative charges
before = len(df_clean)
df_clean = df_clean[
    (df_clean['average_covered_charges'] > 0) &
    (df_clean['average_total_payments'] > 0) &
    (df_clean['Billed amount'] > 0) &
    (df_clean['Medicare payment'] > 0)
]
print(f'  Removed {before - len(df_clean)} zero/negative charge records')

# 2. Remove extreme outliers (> 3 IQR)
before = len(df_clean)
for col in ['average_covered_charges', 'average_total_payments']:
    q1 = df_clean[col].quantile(0.01)
    q99 = df_clean[col].quantile(0.99)
    df_clean = df_clean[(df_clean[col] >= q1) & (df_clean[col] <= q99)]
print(f'  Removed {before - len(df_clean)} extreme outlier records')

# 3. Ensure payments < covered charges (logical constraint)
before = len(df_clean)
df_clean = df_clean[df_clean['average_total_payments'] <= df_clean['average_covered_charges']]
print(f'  Removed {before - len(df_clean)} records where payments exceeded covered charges')

# 4. Standardize state codes
df_clean['provider_state'] = df_clean['provider_state'].str.upper().str.strip()

# 5. Standardize service type
df_clean['service_type'] = df_clean['service_type'].str.lower().str.strip()

# 6. Reset index
df_clean = df_clean.reset_index(drop=True)

print(f'\nFinal cleaned dataset: {len(df_clean):,} records')
print(f'Data retained: {len(df_clean)/len(raw_df)*100:.1f}%')

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution of covered charges
axes[0,0].hist(df_clean['average_covered_charges'], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0,0].set_title('Distribution of Average Covered Charges')
axes[0,0].set_xlabel('Covered Charges ($)')
axes[0,0].set_ylabel('Frequency')

# 2. Distribution of total payments
axes[0,1].hist(df_clean['average_total_payments'], bins=50, color='coral', alpha=0.7, edgecolor='black')
axes[0,1].set_title('Distribution of Average Total Payments')
axes[0,1].set_xlabel('Total Payments ($)')
axes[0,1].set_ylabel('Frequency')

# 3. Records by service type
service_counts = df_clean['service_type'].value_counts()
axes[1,0].bar(range(len(service_counts)), service_counts.values, color='mediumseagreen', alpha=0.8)
axes[1,0].set_xticks(range(len(service_counts)))
axes[1,0].set_xticklabels(service_counts.index, rotation=45, ha='right')
axes[1,0].set_title('Records by Service Type')
axes[1,0].set_ylabel('Count')

# 4. Records by top states
state_counts = df_clean['provider_state'].value_counts().head(15)
axes[1,1].bar(range(len(state_counts)), state_counts.values, color='mediumpurple', alpha=0.8)
axes[1,1].set_xticks(range(len(state_counts)))
axes[1,1].set_xticklabels(state_counts.index)
axes[1,1].set_title('Top 15 States by Record Count')
axes[1,1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('artifacts/01_eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plots saved to artifacts/01_eda_distributions.png')

## 4. Save Cleaned Data

In [ ]:
output_path = 'data/base_cleaned.parquet'
df_clean.to_parquet(output_path, index=False)
print(f'Saved cleaned data to: {output_path}')
print(f'Shape: {df_clean.shape}')
print(f'File size: {os.path.getsize(output_path) / 1024:.1f} KB')
print(f'\nColumns saved:')
for col in df_clean.columns:
    print(f'  - {col}: {df_clean[col].dtype}')
print(f'\n✅ Notebook 1 complete. Run Notebook 2 next.')